# Exercise 3 — next_run_time and seconds_until

Scheduling requires knowing WHEN to act. `next_run_time` computes the next occurrence of a given clock time (today if not yet passed, tomorrow otherwise). `seconds_until` converts that datetime into a sleep duration. Together, they are the core of the scheduler loop.

In [ ]:
import pandas as pd, math, datetime, pathlib, tempfile

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def next_run_time(run_time_str="16:00"):
    """Next local datetime matching run_time_str (HH:MM).

    If the time has not yet passed today, returns today at that time.
    If it has already passed, returns tomorrow at that time.

    Steps:
      1. now = datetime.datetime.now()
      2. h, m = (int(x) for x in run_time_str.split(":"))
      3. target = now.replace(hour=h, minute=m, second=0, microsecond=0)
      4. if target <= now: target += datetime.timedelta(days=1)
      5. return target
    """
    # TODO: ~5 lines
    return datetime.datetime.now()


def seconds_until(target_dt):
    """Seconds from now until target_dt; 0.0 if already past.

    Steps:
      1. delta = target_dt - datetime.datetime.now()
      2. return max(0.0, delta.total_seconds())
    """
    # TODO: 2 lines
    return 0.0


### Checks

In [ ]:
checks = 0

# 1 — next_run_time returns a future datetime
try:
    t = next_run_time("16:00")
    now = datetime.datetime.now()
    assert isinstance(t, datetime.datetime)
    assert t > now, f"expected future time, got {t} vs now {now}"
    checks += 1; print("✅ 1 next_run_time returns a datetime in the future")
except Exception as e:
    print("❌ 1:", e)

# 2 — next_run_time hours and minutes match the requested time
try:
    t = next_run_time("09:30")
    assert t.hour == 9 and t.minute == 30,         f"expected HH=9, MM=30, got {t.hour}:{t.minute}"
    checks += 1; print("✅ 2 next_run_time(09:30) has hour=9, minute=30")
except Exception as e:
    print("❌ 2:", e)

# 3 — past time today → tomorrow
try:
    past_str = "00:01"      # almost certainly already past
    t = next_run_time(past_str)
    now = datetime.datetime.now()
    diff_days = (t.date() - now.date()).days
    assert diff_days <= 1, f"past time should roll to tomorrow, got {diff_days} days ahead"
    checks += 1; print("✅ 3 past time today → scheduled for tomorrow")
except Exception as e:
    print("❌ 3:", e)

# 4 — seconds_until future → positive
try:
    future = datetime.datetime.now() + datetime.timedelta(seconds=60)
    secs   = seconds_until(future)
    assert secs > 0, f"future target should give positive seconds, got {secs}"
    assert secs <= 61, f"should be ~60 seconds, got {secs}"
    checks += 1; print(f"✅ 4 seconds_until(+60s from now) ≈ {secs:.1f}s (positive)")
except Exception as e:
    print("❌ 4:", e)

# 5 — seconds_until past → 0.0
try:
    past = datetime.datetime.now() - datetime.timedelta(seconds=30)
    secs = seconds_until(past)
    assert secs == 0.0, f"past target should return 0.0, got {secs}"
    checks += 1; print("✅ 5 seconds_until(past) == 0.0")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
